# Activity #1: Building a LangGraph Client Agent

So I need to build a LangGraph that acts like a client and makes calls to our A2A server. Basically, this graph will use the A2A protocol to talk to the agent we already have running.

**Before starting:** Make sure the A2A server is running (run `uv run python -m app` in another terminal)



In [67]:
# Let's start by importing what we need
import asyncio
import logging
from typing import Any, Annotated, TypedDict, List
from uuid import uuid4

import httpx

# A2A stuff for talking to our server agent
from a2a.client import A2ACardResolver, A2AClient
from a2a.types import AgentCard, MessageSendParams, SendMessageRequest, SendStreamingMessageRequest
from a2a.utils.constants import AGENT_CARD_WELL_KNOWN_PATH, EXTENDED_AGENT_CARD_PATH

# LangGraph stuff for building our graph
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage

# Set up logging so we can see what's happening
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


## Step 1: Setting Up the Graph State

I need to define what information the graph will keep track of as it runs. This is like the "memory" of our client agent:

- **Messages**: The conversation history (what the user asked, what we got back)
- **A2A Response**: The full response from the A2A server (in case we need it)
- **Task ID and Context ID**: These are for multi-turn conversations - so we can keep talking to the same agent task


In [68]:
class ClientAgentState(TypedDict):
    """This defines what our graph remembers between steps."""
    messages: Annotated[List, add_messages]  # What we've said and what we got back
    a2a_response: Any  # The full response object from A2A server (useful for debugging)
    task_id: str | None  # For keeping track of conversations across multiple turns
    context_id: str | None  # Also for multi-turn - links messages together


## Step 2: Connecting to Our A2A Server

Before we can talk to the A2A server, I need to:
1. Fetch its AgentCard (like its business card - tells us what it can do)
2. Set up the A2AClient to actually send messages to it

This is basically like "introducing ourselves" to the server agent.


In [69]:
# Where our A2A server is running (the one we start with uv run python -m app)
A2A_SERVER_URL = 'http://localhost:10000'

# Keep track of the client so we don't recreate it every time
_httpx_client = None
_a2a_client = None
_agent_card = None

async def initialize_a2a_client() -> tuple[httpx.AsyncClient, A2AClient, AgentCard]:
    """Go fetch the agent card from our server and set up the client."""
    global _httpx_client, _a2a_client, _agent_card
    
    # If we already did this, don't do it again
    if _a2a_client is not None:
        return _httpx_client, _a2a_client, _agent_card
    
    # LLM responses can take a while, so give it some time
    # Also set limits to handle connections better
    httpx_client = httpx.AsyncClient(
        timeout=httpx.Timeout(60.0),
        limits=httpx.Limits(max_connections=10, max_keepalive_connections=5),
        http2=False  # Use HTTP/1.1 which is more reliable for localhost
    )
    
    # This resolver helps us get the agent card from the server
    resolver = A2ACardResolver(
        httpx_client=httpx_client,
        base_url=A2A_SERVER_URL,
    )
    
    # Try to get the agent card
    logger.info(f'Fetching agent card from {A2A_SERVER_URL}')
    try:
        agent_card = await resolver.get_agent_card()
        logger.info(f'Got it! Agent name: {agent_card.name}')
        
        # Now create the client that can actually send messages
        a2a_client = A2AClient(httpx_client=httpx_client, agent_card=agent_card)
        
        # Save these for later use
        _httpx_client = httpx_client
        _a2a_client = a2a_client
        _agent_card = agent_card
        
        return httpx_client, a2a_client, agent_card
    except Exception as e:
        logger.error(f'Oops, something went wrong: {e}')
        raise RuntimeError(f'Could not connect to A2A server at {A2A_SERVER_URL}. Did you start the server?') from e

# Let's connect to the server now!
httpx_client, a2a_client, agent_card = await initialize_a2a_client()
print(f"✅ Connected to: {agent_card.name}")
print(f"   What it does: {agent_card.description}")
print(f"   Tools it has: {[skill.name for skill in agent_card.skills]}")


INFO:__main__:Fetching agent card from http://localhost:10000
INFO:httpx:HTTP Request: GET http://localhost:10000/.well-known/agent-card.json "HTTP/1.1 200 OK"
INFO:a2a.client.card_resolver:Successfully fetched agent card data from http://localhost:10000/.well-known/agent-card.json: {'capabilities': {'pushNotifications': True, 'streaming': True}, 'defaultInputModes': ['text', 'text/plain'], 'defaultOutputModes': ['text', 'text/plain'], 'description': 'A helpful AI assistant with web search, academic paper search, and document retrieval capabilities', 'name': 'General Purpose Agent', 'preferredTransport': 'JSONRPC', 'protocolVersion': '0.3.0', 'skills': [{'description': 'Search the web for current information', 'examples': ['What are the latest news about AI?'], 'id': 'web_search', 'name': 'Web Search Tool', 'tags': ['search', 'web', 'internet']}, {'description': 'Search for academic papers on arXiv', 'examples': ['Find recent papers on large language models'], 'id': 'arxiv_search', 'na

✅ Connected to: General Purpose Agent
   What it does: A helpful AI assistant with web search, academic paper search, and document retrieval capabilities
   Tools it has: ['Web Search Tool', 'Academic Paper Search', 'Document Retrieval']


/var/folders/7f/yj7mwjbd2kdg8fnh6p0k8y4w0000gp/T/ipykernel_42887/3700458333.py:38: DeprecationWarning: A2AClient is deprecated and will be removed in a future version. Use ClientFactory to create a client with a JSON-RPC transport.
  a2a_client = A2AClient(httpx_client=httpx_client, agent_card=agent_card)


## Step 3: Building the Graph Nodes

Now I need to create the actual "steps" in our graph. Think of nodes as functions that do specific things:

1. **Call A2A Server**: This is where we actually talk to the server agent
2. **Format Response**: Clean up the response we get back so it's nice to read

Each node takes the current state, does something with it, and returns an updated state.


In [70]:
async def call_a2a_server(state: ClientAgentState) -> dict[str, Any]:
    """This node actually sends a message to our A2A server agent."""
    global a2a_client
    
    # Get what the user asked from the last message
    last_message = state["messages"][-1]
    if isinstance(last_message, HumanMessage):
        user_query = last_message.content
    else:
        user_query = str(last_message.content)
    
    logger.debug(f"Sending to A2A server: {user_query[:100]}...")
    
    # Package up the message in the format A2A expects
    send_message_payload: dict[str, Any] = {
        'message': {
            'role': 'user',
            'parts': [{'kind': 'text', 'text': user_query}],
            'message_id': uuid4().hex,  # Give each message a unique ID
        },
    }
    
    # If this is part of a longer conversation, include the context
    if state.get('task_id') and state.get('context_id'):
        send_message_payload['message']['task_id'] = state['task_id']
        send_message_payload['message']['context_id'] = state['context_id']
        logger.info("🔄 This is a follow-up message...")
    
    # Create the request object
    request = SendMessageRequest(
        id=str(uuid4()),
        params=MessageSendParams(**send_message_payload)
    )
    
    try:
        # Use streaming since that's what the server does - much simpler!
        streaming_request = SendStreamingMessageRequest(
            id=str(uuid4()),
            params=MessageSendParams(**send_message_payload)
        )
        
        # Collect the streaming response
        # We'll collect all text and then filter to get the final answer
        all_text_chunks = []
        task_id = None
        context_id = None
        
        stream_response = a2a_client.send_message_streaming(streaming_request)
        chunk_count = 0
        all_chunks_raw = []  # Store all raw chunks for debugging
        
        # Wait for all chunks to come through - the final answer is in later chunks
        logger.debug("Waiting for streaming response...")
        
        async for chunk in stream_response:
            chunk_count += 1
            # Store raw chunk for later inspection
            all_chunks_raw.append((chunk_count, chunk))
            try:
                # Convert chunk to dict - handle both dict and object cases
                if isinstance(chunk, dict):
                    chunk_dict = chunk
                elif hasattr(chunk, 'model_dump'):
                    chunk_dict = chunk.model_dump(mode='json', exclude_none=True)
                else:
                    # Fallback: try to convert to dict
                    chunk_dict = dict(chunk) if hasattr(chunk, '__dict__') else {}
                
                # Extract chunk info for processing (but don't log full structure)
                result = chunk_dict.get('result', {})
                is_final = result.get('final', False)
                kind = result.get('kind', '')
                
                # Only log important milestones (not full chunk structure)
                if is_final:
                    state = result.get('status', {}).get('state', 'N/A')
                    logger.info(f"🔚 Received final chunk (state: {state})")
                elif kind == 'artifact-update':
                    logger.info(f"📦 Received artifact chunk {chunk_count}")
                # Otherwise, just process silently
                
                # Extract text from chunk - navigate through the structure safely
                if isinstance(chunk_dict, dict):
                    # Chunk might have result.message.parts.text or other structures
                    result = chunk_dict.get('result', {})
                    
                    # If result is not a dict, skip it (don't try to access attributes)
                    if not isinstance(result, dict):
                        # Check if it's an object we shouldn't touch
                        if hasattr(result, '__class__'):
                            logger.debug(f"Skipping non-dict result of type: {result.__class__.__name__}")
                        result = {}
                    
                    if result and isinstance(result, dict):
                        # Get task info from first chunk
                        if task_id is None:
                            task_id = result.get('id') or result.get('taskId') or state.get('task_id')
                        if context_id is None:
                            context_id = result.get('contextId') or state.get('context_id')
                        
                        # Get message text - collect ALL text from chunks, we'll filter later
                        is_final = result.get('final', False)
                        kind = result.get('kind', '')
                        
                        # Path 1: status-update chunks - collect all message text
                        status = result.get('status', {})
                        if status and isinstance(status, dict):
                            message = status.get('message', {})
                            if message and isinstance(message, dict):
                                parts = message.get('parts', [])
                                for part in parts:
                                    if isinstance(part, dict) and 'text' in part:
                                        text = part['text'].strip()
                                        if text:  # Only add non-empty text
                                            all_text_chunks.append({
                                                'text': text,
                                                'final': is_final,
                                                'state': status.get('state', ''),
                                                'chunk_num': chunk_count
                                            })
                        
                        # Path 2: task chunks history - collect agent messages
                        if 'history' in result:
                            history = result.get('history', [])
                            for hist_item in history:
                                if isinstance(hist_item, dict) and hist_item.get('role') == 'agent':
                                    parts = hist_item.get('parts', [])
                                    for part in parts:
                                        if isinstance(part, dict) and 'text' in part:
                                            text = part['text'].strip()
                                            if text:
                                                all_text_chunks.append({
                                                    'text': text,
                                                    'final': is_final,
                                                    'kind': kind,
                                                    'chunk_num': chunk_count
                                                })
                        
                        # Path 3: Check artifact (singular) - artifact-update chunks have result.artifact (not artifacts!)
                        # THIS IS WHERE THE ACTUAL ANSWER IS!
                        if 'artifact' in result:
                            artifact = result.get('artifact', {})
                            if isinstance(artifact, dict) and 'parts' in artifact:
                                for part in artifact['parts']:
                                    if isinstance(part, dict):
                                        text = None
                                        if 'text' in part:
                                            text = part['text'].strip()
                                        elif 'root' in part and isinstance(part['root'], dict) and 'text' in part['root']:
                                            text = part['root']['text'].strip()
                                        if text:
                                            logger.debug(f"Found artifact text in chunk {chunk_count} ({len(text)} chars)")
                                            all_text_chunks.append({
                                                'text': text,
                                                'final': is_final,
                                                'source': 'artifact',
                                                'chunk_num': chunk_count
                                            })
                        
                        # Path 3b: Check artifacts (plural) - some chunks might have artifacts array
                        if 'artifacts' in result:
                            artifacts = result.get('artifacts', [])
                            logger.debug(f"Found {len(artifacts)} artifacts array in chunk {chunk_count}")
                            for artifact in artifacts:
                                if isinstance(artifact, dict) and 'parts' in artifact:
                                    for part in artifact['parts']:
                                        if isinstance(part, dict):
                                            text = None
                                            if 'text' in part:
                                                text = part['text'].strip()
                                            elif 'root' in part and isinstance(part['root'], dict) and 'text' in part['root']:
                                                text = part['root']['text'].strip()
                                            if text:
                                                logger.info(f"✅ Found artifact text in chunk {chunk_count}: {text[:100]}...")
                                                all_text_chunks.append({
                                                    'text': text,
                                                    'final': is_final,
                                                    'source': 'artifacts',
                                                    'chunk_num': chunk_count
                                                })
                        
                        # Path 4: direct message.parts (for any chunks)
                        message = result.get('message', {})
                        if message and isinstance(message, dict):
                            parts = message.get('parts', [])
                            for part in parts:
                                if isinstance(part, dict) and 'text' in part:
                                    text = part['text'].strip()
                                    if text:
                                        all_text_chunks.append({
                                            'text': text,
                                            'final': is_final,
                                            'source': 'direct',
                                            'chunk_num': chunk_count
                                        })
                        

            except Exception as chunk_error:
                # Log but don't stop - just continue processing chunks
                logger.warning(f"Error processing chunk {chunk_count}: {chunk_error}")
                continue
        
        logger.info(f"📦 Processed {chunk_count} chunks, found {len(all_text_chunks)} text chunks")
        
        # Use task_id and context_id from state if we didn't get them
        if task_id is None:
            task_id = state.get('task_id')
        if context_id is None:
            context_id = state.get('context_id')
        
        # Now filter to get the ACTUAL ANSWER
        response_text = ""
        
        # Filter out intermediate status messages
        meaningful_chunks = [
            c for c in all_text_chunks 
            if not any(skip in c['text'].lower() for skip in ['searching', 'processing', 'working'])
        ]
        
        # Priority: artifact/artifacts > meaningful chunks > all chunks
        # Artifacts are where the final answer usually is!
        artifact_chunks = [c for c in all_text_chunks if c.get('source') in ['artifact', 'artifacts']]
        if artifact_chunks:
            # Use artifacts - they contain the final answer
            best_chunk = max(artifact_chunks, key=lambda x: x.get('chunk_num', 0))
            response_text = best_chunk['text']
        elif meaningful_chunks:
            # Strategy: Get the LAST meaningful chunk OR the LONGEST one
            # The actual answer usually comes in later chunks and is longer
            
            # Get the latest chunk (answer comes after status updates)
            latest_chunk = max(meaningful_chunks, key=lambda x: x.get('chunk_num', 0))
            
            # Get the longest chunk (real answers are usually much longer)
            longest_chunk = max(meaningful_chunks, key=lambda x: len(x['text']))
            
            # Use longest if it's significantly longer (probably the real answer)
            # Or use latest if they're similar length
            if len(longest_chunk['text']) > len(latest_chunk['text']) * 1.5:
                best_chunk = longest_chunk
            else:
                best_chunk = latest_chunk
            
            response_text = best_chunk['text']
        
        # Fallback: use the longest chunk even if it contains status words
        elif all_text_chunks:
            longest = max(all_text_chunks, key=lambda x: len(x['text']))
            response_text = longest['text']
        
        # If we got task_id but no response text, try using non-streaming API to get the final result
        # The non-streaming response might have the artifacts
        if not response_text and task_id:
            logger.debug(f"No text in streaming response. Trying non-streaming API...")
            try:
                # Use non-streaming send_message to get the complete response with artifacts
                non_streaming_request = SendMessageRequest(
                    id=str(uuid4()),
                    params=MessageSendParams(**send_message_payload)
                )
                final_response = await a2a_client.send_message(non_streaming_request)
                
                # Extract from the complete response
                final_dict = final_response.model_dump(mode='json', exclude_none=True)
                result_dict = final_dict.get('root', {}).get('result', {})
                
                # Check artifacts in the complete response
                artifacts = result_dict.get('artifacts', [])
                if artifacts:
                    for artifact in artifacts:
                        if isinstance(artifact, dict) and 'parts' in artifact:
                            for part in artifact['parts']:
                                if isinstance(part, dict):
                                    text = None
                                    if 'text' in part:
                                        text = part['text'].strip()
                                    elif 'root' in part and isinstance(part['root'], dict) and 'text' in part['root']:
                                        text = part['root']['text'].strip()
                                    if text:
                                        response_text += text + " "
            except Exception as e:
                logger.debug(f"Non-streaming API fallback failed: {e}")
        
        # Clean up
        response_text = response_text.strip()
        
        # Log result
        if not response_text:
            logger.warning(f"⚠️ No text extracted from {chunk_count} chunks!")
        else:
            logger.info(f"✅ Response received ({len(response_text)} chars)")
        
        # Update our state
        return {
            "messages": [AIMessage(content=response_text)],
            "a2a_response": {"streaming": True, "text_length": len(response_text)},  # Simplified
            "task_id": task_id,
            "context_id": context_id,
        }
    except Exception as e:
        logger.error(f"❌ Something broke: {e}")
        error_msg = f"Error communicating with A2A server: {str(e)}"
        return {
            "messages": [AIMessage(content=error_msg)],
            "a2a_response": None,
        }


def format_response(state: ClientAgentState) -> dict[str, Any]:
    """This node just makes sure the response is formatted nicely (it's already pretty good though)."""
    # The response is already in messages, so we don't need to change anything
    return {}


## Step 4: Putting It All Together - Build the Graph

Now I'll wire up the nodes I just created. This is like drawing arrows between the steps to show how data flows through our graph.


In [71]:
def build_client_agent_graph():
    """Put together our graph with the nodes we created."""
    graph = StateGraph(ClientAgentState)
    
    # Add the two nodes we just made
    graph.add_node("call_a2a", call_a2a_server)
    graph.add_node("format_response", format_response)
    
    # Where the graph starts when we run it
    graph.set_entry_point("call_a2a")
    
    # Connect the nodes: first call A2A, then format, then we're done
    graph.add_edge("call_a2a", "format_response")
    graph.add_edge("format_response", END)
    
    # Compile it so we can actually use it
    return graph.compile()

# Let's build it!
client_graph = build_client_agent_graph()
print("✅ Graph is ready to go!")


✅ Graph is ready to go!


## Step 5: Time to Test It!

Let's see if our graph actually works! I'll create a helper function to make testing easier, then try it with a real query.


In [72]:
# Helper function to make testing easier
async def run_client_agent(query: str):
    """Just feed it a question and it'll run the whole graph."""
    # Start with the user's question
    initial_state = {
        "messages": [HumanMessage(content=query)],
        "a2a_response": None,
        "task_id": None,
        "context_id": None,
    }
    
    # Run the graph!
    print(f"🔍 Asking: {query}")
    print("─" * 60)
    
    result = await client_graph.ainvoke(initial_state)
    
    # Show what we got back
    if result["messages"]:
        last_message = result["messages"][-1]
        if isinstance(last_message, AIMessage):
            print(f"\n📋 Here's what the A2A agent said:")
            print("─" * 60)
            print(last_message.content)
            print("─" * 60)
    
    return result

# Let's try it out with a question!
test_query = "What are the latest developments in artificial intelligence?"
result = await run_client_agent(test_query)


INFO:httpx:HTTP Request: POST http://localhost:10000/ "HTTP/1.1 200 OK"


🔍 Asking: What are the latest developments in artificial intelligence?
────────────────────────────────────────────────────────────


INFO:__main__:📦 Received artifact chunk 4
INFO:__main__:🔚 Received final chunk (state: completed)
INFO:__main__:📦 Processed 5 chunks, found 3 text chunks
INFO:__main__:✅ Response received (1514 chars)



📋 Here's what the A2A agent said:
────────────────────────────────────────────────────────────
Recent developments in artificial intelligence include several exciting advancements:

1. AI is increasingly becoming a scientific collaborator. Systems like DeepMind’s Co-Scientist and Stanford’s Virtual Lab are autonomously generating, testing, and validating hypotheses. In biology, models like Profluent’s ProGen3 have demonstrated that scaling laws apply to proteins.

2. Structured reasoning has entered the physical world through "Chain-of-Action" planning. Embodied AI systems such as AI2’s Molmo-Act and Google’s Gemini Robotics 1.5 can reason step-by-step before acting.

3. The latest generation of AI models, such as Optimus, show improved physical capabilities including balancing and object manipulation, with broader deployment targets in industrial settings.

4. AI safety research is evolving, with models now able to imitate alignment under supervision, sparking debates about transpare